# Predicción de Abandono de Clientes (Telco Customer Churn)
## Notebook 2 de 2 Modelamiento supervisado, no supervisado y evaluación

**MLY1101 Machine Learning | Caso A | Evaluación Parcial N°2**

Continúa el notebook `01_EDA_Preparacion_Churn.ipynb` y desarrolla las fases 4 y 5 de CRISP-DM (Modelamiento y Evaluación) y una propuesta de Despliegue (fase 6).

## 1. Flujo de trabajo (CRISP-DM)

1. **Negocio y datos** (notebook 1): problema, KPIs, origen y calidad de datos, EDA.
2. **Preparación** (notebook 1 + Pipeline aquí): dataset limpio y codificado; el escalado se aplica **dentro** de cada `Pipeline`, ajustándose solo con datos de entrenamiento.
3. **Modelamiento**: línea base → Regresión Logística → Random Forest, con búsqueda de hiperparámetros por validación cruzada estratificada (5 particiones) **solo sobre train**.
4. **Evaluación**: el conjunto de prueba (20%) se usa una sola vez al final; el umbral de decisión se elige con predicciones fuera de muestra de train (no con test).
5. **No supervisado**: segmentación K-Means para diseñar campañas por perfil.
6. **Despliegue** (propuesta): Sección 10.

## 2. Librerías y carga de datos

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import joblib

from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_validate, cross_val_predict)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, precision_recall_curve, silhouette_score, adjusted_rand_score)

pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.titleweight": "bold"})
RANDOM_STATE = 42
COLOR_NO, COLOR_SI = "#2a78d6", "#eb6834"
C_LR, C_RF = "#2a78d6", "#1baf7a"
SEG_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
IMG, MODELS = "../images/", "../models/"

In [ ]:
df = pd.read_csv("../data/telco_churn_limpio.csv")   # generado por el notebook 1
y = df["Churn"]
X_all = df.drop(columns=["Churn"])

# Misma partición que el notebook 1 (80/20 estratificada, misma semilla)
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"Train: {len(X_train_all)} ({y_train.mean():.1%} churn) | Test: {len(X_test_all)} ({y_test.mean():.1%} churn)")

## 3. Selección de variables: ¿se usan los atributos demográficos protegidos?

El EDA mostró que `gender` no se asocia al churn y que `SeniorCitizen` sí, pero en parte porque los adultos mayores contratan planes más caros. Antes de modelar se prueba con validación cruzada (configuración fija, solo train) si excluir estas variables cuesta desempeño. Si no cuesta, se excluyen: así el modelo no puede discriminar directamente por género o edad, y ambas variables quedan solo para **auditar** el resultado (Sección 8).

In [ ]:
NUM_MINMAX, NUM_ROBUST = ["tenure", "MonthlyCharges"], ["TotalCharges"]

def preprocesador():
    # Escalado elegido en el notebook 1 según la forma de cada variable; el resto (0/1) pasa sin cambios
    return ColumnTransformer([("minmax", MinMaxScaler(), NUM_MINMAX),
                              ("robust", RobustScaler(), NUM_ROBUST)], remainder="passthrough")

conjuntos = {
    "Todas las variables": list(X_all.columns),
    "Sin gender": [c for c in X_all.columns if c != "gender_Female"],
    "Sin gender ni SeniorCitizen": [c for c in X_all.columns if c not in ("gender_Female", "SeniorCitizen")],
}
METRICAS_CV = ["recall", "precision", "f1", "roc_auc", "average_precision"]
filas = []
for nombre, cols in conjuntos.items():
    for mod_nombre, modelo in [
        ("Reg. Logística", LogisticRegression(class_weight="balanced", max_iter=2000, solver="liblinear")),
        ("Random Forest", RandomForestClassifier(n_estimators=200, min_samples_leaf=10, class_weight="balanced",
                                                 random_state=RANDOM_STATE, n_jobs=-1))]:
        pipe = Pipeline([("prep", preprocesador()), ("modelo", modelo)])
        r = cross_validate(pipe, X_train_all[cols], y_train, cv=cv, scoring=METRICAS_CV)
        filas.append([nombre, mod_nombre, len(cols)] + [r[f"test_{m}"].mean() for m in METRICAS_CV])
comp_vars = pd.DataFrame(filas, columns=["Conjunto", "Modelo", "n_variables"] + METRICAS_CV).round(3)
comp_vars

**Observación:** excluir `gender` y `SeniorCitizen` **no cambia el desempeño**: la Regresión Logística mantiene F1 = 0,629 y PR-AUC = 0,659 en los tres conjuntos, y el Random Forest pasa de F1 0,635 a 0,633 y de PR-AUC 0,662 a 0,664. Las diferencias (≤ 0,002) son menores que la variación entre particiones. **Decisión:** se entrenan los modelos **sin** estas dos variables. Se gana en ética (el modelo no puede usar directamente género ni edad) sin perder capacidad predictiva, y ambas se guardan solo para la auditoría de sesgo.

In [ ]:
FEATURES = conjuntos["Sin gender ni SeniorCitizen"]
X_train, X_test = X_train_all[FEATURES], X_test_all[FEATURES]
print(f"{len(FEATURES)} variables predictoras:", FEATURES)

## 4. Línea base (modelo trivial)

Un modelo que siempre predice la clase mayoritaria ("No abandona") fija el piso que cualquier modelo útil debe superar.

In [ ]:
def metricas(y_true, y_pred, y_score=None):
    m = {"Accuracy": accuracy_score(y_true, y_pred),
         "Recall": recall_score(y_true, y_pred, zero_division=0),
         "Precisión": precision_score(y_true, y_pred, zero_division=0),
         "F1": f1_score(y_true, y_pred, zero_division=0)}
    if y_score is not None:
        m["ROC-AUC"] = roc_auc_score(y_true, y_score)
        m["PR-AUC"] = average_precision_score(y_true, y_score)
    return m

base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
res_base = metricas(y_test, base.predict(X_test), base.predict_proba(X_test)[:, 1])
pd.Series(res_base).round(3)

**Observación:** la línea base obtiene **73,5% de accuracy** sin detectar a **ningún** cliente que se va (recall 0). Confirma que la accuracy es engañosa con clases desbalanceadas; los modelos se comparan por **recall, precisión, F1 y PR-AUC** (área bajo la curva precisión-recall, que se centra en la clase minoritaria; su piso es la tasa de churn, 0,265).

## 5. Modelos supervisados

Problema de **clasificación binaria** (abandona / no abandona) con probabilidad de salida, para poder ordenar clientes por riesgo.

### 5.1 Modelo 1: Regresión Logística

**Por qué este algoritmo:**
- Modela directamente la **probabilidad** de churn y sus coeficientes son **interpretables** (cuánto sube o baja el riesgo cada característica), algo necesario para explicarle al área comercial por qué un cliente está en riesgo.
- Es el modelo con mejor recall en los estudios previos sobre este mismo dataset (Zhu 2023; Wanda 2024; ver notebook 1).
- Es sensible a la escala de las variables ⇒ usa el escalado justificado en el notebook 1 (MinMax / Robust).

**Configuración:**
- `class_weight="balanced"`: penaliza más los errores en la clase minoritaria (churn), compensando el desbalance 1 : 2,8 sin generar datos sintéticos.
- Hiperparámetros buscados: `C` (inverso de la regularización: valores bajos = modelo más simple, controla el sobreajuste y la colinealidad entre `tenure` y `TotalCharges`) y `penalty` (`l2` reparte el peso entre variables correlacionadas; `l1` puede dejar coeficientes en cero y seleccionar variables).
- Criterio de búsqueda: **F1** (equilibrio entre recall y precisión; optimizar solo recall llevaría a marcar a todos como riesgo).

In [ ]:
pipe_lr = Pipeline([("prep", preprocesador()),
                    ("modelo", LogisticRegression(class_weight="balanced", max_iter=5000, solver="liblinear"))])
grid_lr = GridSearchCV(pipe_lr, {"modelo__C": [0.01, 0.1, 1, 10], "modelo__penalty": ["l1", "l2"]},
                       scoring="f1", cv=cv, n_jobs=-1).fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_lr.best_params_, "| F1 CV:", round(grid_lr.best_score_, 4))
pd.DataFrame(grid_lr.cv_results_)[["param_modelo__C", "param_modelo__penalty", "mean_test_score", "std_test_score"]] \
  .sort_values("mean_test_score", ascending=False).round(4).head(8)

**Observación:** la mejor configuración es `C = 1` con penalización `l1` (F1 en validación 0,631). Las 8 combinaciones probadas quedan entre 0,624 y 0,631, una diferencia menor que la desviación entre particiones (≈ 0,02): el modelo es **poco sensible** a la regularización, señal de que no está sobreajustando. La penalización `l1` dejó en cero el coeficiente de `MonthlyCharges` (ver Sección 7): su información ya está contenida en los servicios contratados (fibra, streaming, etc.), que son los que determinan el cargo.

### 5.2 Modelo 2: Random Forest

**Por qué este algoritmo:**
- Captura **relaciones no lineales e interacciones** sin especificarlas. El EDA mostró una interacción clara: el cargo mensual alto aumenta el churn solo cuando va con contrato mes a mes (deciles, notebook 1). Una regresión logística no la modela salvo que se agregue a mano.
- Es un ensamble de árboles con *bagging*: promedia muchos árboles entrenados con muestras distintas, lo que reduce la varianza y el sobreajuste de un árbol individual.
- Entrega importancia de variables para interpretar.
- No necesita escalado; se mantiene el mismo `Pipeline` para comparar ambos modelos en igualdad de condiciones (el escalado no altera los cortes de los árboles).

**Configuración:**
- `n_estimators=300` árboles (suficiente para estabilizar el promedio), `class_weight="balanced"`.
- Hiperparámetros buscados: `max_depth` (profundidad máxima), `min_samples_leaf` (mínimo de clientes por hoja; valores altos evitan reglas basadas en pocos clientes) y `max_features` (variables candidatas en cada corte; menos variables = árboles más distintos entre sí).

In [ ]:
pipe_rf = Pipeline([("prep", preprocesador()),
                    ("modelo", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                      random_state=RANDOM_STATE, n_jobs=-1))])
grid_rf = GridSearchCV(pipe_rf, {"modelo__max_depth": [None, 6, 10],
                                 "modelo__min_samples_leaf": [1, 5, 10, 20],
                                 "modelo__max_features": ["sqrt", 0.5]},
                       scoring="f1", cv=cv, n_jobs=-1).fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_rf.best_params_, "| F1 CV:", round(grid_rf.best_score_, 4))
pd.DataFrame(grid_rf.cv_results_)[["param_modelo__max_depth", "param_modelo__min_samples_leaf",
                                   "param_modelo__max_features", "mean_test_score", "std_test_score"]] \
  .sort_values("mean_test_score", ascending=False).round(4).head(8)

In [ ]:
# Sobreajuste: F1 en train vs F1 en validación cruzada para distintos min_samples_leaf (resto en su mejor valor)
cvr = pd.DataFrame(GridSearchCV(pipe_rf, {"modelo__max_depth": [grid_rf.best_params_["modelo__max_depth"]],
                                          "modelo__min_samples_leaf": [1, 5, 10, 20],
                                          "modelo__max_features": [grid_rf.best_params_["modelo__max_features"]]},
                                scoring="f1", cv=cv, n_jobs=-1, return_train_score=True).fit(X_train, y_train).cv_results_)
cvr[["param_modelo__min_samples_leaf", "mean_train_score", "mean_test_score"]].round(3) \
   .rename(columns={"param_modelo__min_samples_leaf": "min_samples_leaf", "mean_train_score": "F1 train",
                    "mean_test_score": "F1 validación"})

**Observación:** la mejor configuración es `max_depth = 10`, `max_features = 0.5` y `min_samples_leaf = 10` (F1 en validación 0,637). La tabla de sobreajuste justifica el `min_samples_leaf`: con hojas de 1 cliente el bosque memoriza train (F1 train 0,809 vs validación 0,629, brecha de 0,18); con hojas de 10 clientes la brecha baja a 0,07 (0,708 vs 0,637) y el F1 de validación es el más alto. Limitar la profundidad y el tamaño de hoja **controla el sobreajuste**.

### 5.3 Comparación en validación cruzada (estabilidad)

Se evalúan ambos modelos ya ajustados con las mismas 5 particiones de train. Se reporta media y desviación estándar: un modelo con menor desviación es más estable ante cambios en los datos.

In [ ]:
modelos = {"Regresión Logística": grid_lr.best_estimator_, "Random Forest": grid_rf.best_estimator_}
cv_res = {n: cross_validate(m, X_train, y_train, cv=cv, scoring=METRICAS_CV) for n, m in modelos.items()}
tabla_cv = pd.DataFrame({n: {m: f"{r['test_'+m].mean():.3f} ± {r['test_'+m].std():.3f}" for m in METRICAS_CV}
                         for n, r in cv_res.items()}).T
tabla_cv

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, m, et in zip(axes, ["recall", "precision", "f1", "average_precision"], ["Recall", "Precisión", "F1", "PR-AUC"]):
    datos = [cv_res[n][f"test_{m}"] for n in modelos]
    bp = ax.boxplot(datos, tick_labels=["Reg. Log.", "Random F."], patch_artist=True, widths=0.5,
                    medianprops={"color": "black"})
    for p, c in zip(bp["boxes"], [C_LR, C_RF]): p.set_facecolor(c); p.set_alpha(0.7)
    ax.set_title(et)
fig.suptitle("Gráfico 6. Métricas en las 5 particiones de validación cruzada (train)", fontweight="bold")
plt.tight_layout(); plt.savefig(IMG + "g6_cv_comparacion.png", dpi=130); plt.show()

**Observación:** en validación cruzada los modelos quedan prácticamente empatados y con perfiles distintos:
- **Regresión Logística:** más **recall** (0,803 ± 0,039): detecta más clientes que se van.
- **Random Forest:** más **precisión** (0,541 vs 0,520) y algo más de F1 (0,637 vs 0,631) y PR-AUC (0,664 vs 0,660). Además es **más estable**: su desviación en recall (± 0,021) es la mitad que la de la Regresión Logística (± 0,039).
- ROC-AUC prácticamente igual (0,846 vs 0,845).

Todas las diferencias entre medias son menores que la desviación entre particiones: ningún modelo es claramente superior.